# World Cup Elo Predictor
Despliegue completo desde SageMaker Notebook.

Ejecutar las celdas **en orden**.

### 0. Verificar entorno

In [ ]:
import sys, json, boto3
print(f"Python {sys.version}")
print(f"Region: {boto3.session.Session().region_name}")
print(f"Account: {boto3.client('sts').get_caller_identity()['Account']}")

### 1. Instalar dependencias

In [ ]:
!pip install -q requests beautifulsoup4 numpy

### 2. Extraer ratings Elo actuales

In [ ]:
%run scripts/fetch_ratings.py

### 3. Desplegar infraestructura (SOLO PRIMERA VEZ)

Crea S3, IAM, Lambda, API Gateway y endpoint SageMaker.
**Tarda ~8 minutos** (por la creacion del endpoint).

In [ ]:
%run scripts/deploy.py

### 4. Probar el endpoint

In [ ]:
sm = boto3.client("sagemaker-runtime")
resp = sm.invoke_endpoint(
    EndpointName="worldcup-elo-endpoint",
    ContentType="application/json",
    Body=json.dumps({"team_a": "Mexico", "team_b": "South Africa"}),
)
print(json.loads(resp["Body"].read()))

### 5. Abrir la UI

La URL de la UI se imprimio al final del paso 3.
Si no la ves, ejecuta:

In [ ]:
state = json.loads(open(".deploy_state.json").read())
print(f"UI: {state.get('ui_url', 'N/A')}")
print(f"API: {state.get('api_url', 'N/A')}")

---
### Actualizar datos (futuras ejecuciones)

Cuando quieras refrescar los ratings Elo y actualizar el modelo:

In [ ]:
%run scripts/refresh.py

### Limpiar (cuando termine el mundial)

In [ ]:
# Eliminar endpoint SageMaker
sagemaker = boto3.client('sagemaker')
sagemaker.delete_endpoint(EndpointName='worldcup-elo-endpoint')

# Eliminar bucket S3 (vacia primero)
state = json.loads(open('.deploy_state.json').read())
s3 = boto3.client('s3')
objects = s3.list_objects_v2(Bucket=state['bucket'])
if 'Contents' in objects:
    s3.delete_objects(Bucket=state['bucket'], Delete={'Objects': [{'Key': o['Key']} for o in objects['Contents']]})
s3.delete_bucket(Bucket=state['bucket'])
print("Limpieza completada.")